# M2 Best $+$ Focal BCE $+$ Augmentation $+$ SupCon

Kế thừa config M2 Best (LR $2\times10^{-5}$, WD 0,05, dropout 0,3, 10 epoch), thêm Focal BCE, augmentation giữ nhãn và SupCon. Baseline cần vượt: M2 Best F1 0,9707.


In [ ]:
import base64, io, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path
WORK = Path('/kaggle/working')
CLEAN = WORK / 'clean_m2p'
CLEAN.mkdir(parents=True, exist_ok=True)

def sh(cmd):
    print(f'\n$ {cmd}', flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    with (WORK / 'live_progress.log').open('a', encoding='utf-8') as lf:
        for line in iter(p.stdout.readline, ''):
            print(line, end='', flush=True)
            lf.write(line); lf.flush()
    p.stdout.close()
    rc = p.wait()
    if rc != 0: raise RuntimeError(f'Fail rc={rc}: {cmd}')


In [ ]:
payload = '...'
repo_root = WORK / 'project'
repo_root.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(payload)), mode='r:gz') as tar:
    tar.extractall(repo_root)
os.chdir(repo_root)
print('Repo:', os.getcwd(), flush=True)


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), flush=True)
if not torch.cuda.is_available(): raise SystemError('Chua bat GPU!')
print('GPU:', torch.cuda.get_device_name(0), flush=True)
sh('pip install -q -U transformers datasets scikit-learn scipy pyarrow \"tree-sitter>=0.23.0\" \"tree-sitter-java>=0.23.5\"')


In [ ]:
data_raw = repo_root / 'data/raw/bigclonebench'
data_raw.mkdir(parents=True, exist_ok=True)
for name in ('bcb_train.parquet', 'bcb_validation.parquet', 'bcb_test.parquet'):
    cands = list(Path('/kaggle/input').rglob(name))
    dest = data_raw / name
    if cands and not dest.exists(): sh(f'cp "{cands[0]}" "{dest}"')
    print(name, dest.exists(), flush=True)
    assert dest.exists(), f'Thieu {name}'
sh('ls -lh data/raw/bigclonebench/*.parquet')


In [ ]:
cfg = 'configs/tuning/exp_m2_best_plus.yaml'
print('=' * 70)
print('>>> BAT DAU M2 BEST + FOCAL + AUG + SUPCON <<<')
print('=' * 70)
t0 = time.perf_counter()
sh(f'python -m src.experiments.bcb_runner --config {cfg} --mode preflight --device cuda')
sh(f'python -m src.experiments.bcb_runner --config {cfg} --mode train --device cuda --time-budget-s 18000')
sh(f'python -m src.experiments.bcb_runner --config {cfg} --mode evaluate --device cuda')
print(f'>>> XONG TRONG {(time.perf_counter()-t0)/60:.1f} PHUT', flush=True)
print((repo_root / 'results/exp_m2_best_plus/test_metrics.json').read_text(encoding='utf-8'), flush=True)


In [ ]:
res_dir = repo_root / 'results/exp_m2_best_plus'
for p in res_dir.glob('*'):
    if p.is_file() and p.suffix in ('.pt', '.json'):
        name = 'm2p_' + p.name
        shutil.copy2(p, CLEAN / name)
        shutil.copy2(p, WORK / name)
        print('  [SAVED]', name, flush=True)
log = WORK / 'live_progress.log'
if log.exists():
    shutil.copy2(log, CLEAN / 'm2p_live_progress.log')
    shutil.copy2(log, WORK / 'm2p_live_progress.log')
with tarfile.open(WORK / 'm2_best_plus_bundle.tar.gz', 'w:gz') as tar:
    for f in CLEAN.glob('*'): tar.add(f, arcname=f.name)
os.chdir(WORK)
for junk in ('project', 'data', 'clean_m2p'):
    shutil.rmtree(WORK / junk, ignore_errors=True)
print('>>> XONG 100%!', flush=True)